# 2次元イジング模型：厳密解（オンサガーの解）との詳細比較ガイド

このノートブックでは、シミュレーション結果と理論的な「正解（厳密解）」を比較する方法について詳しく解説します。なぜビンダー累積量が必要なのか、そしてシミュレーションデータがどのように理論曲線に近づいていくのかを深く理解することを目的とします。

---

## 1. 理論的な正解：オンサガーの厳密解

1944年、ラーシュ・オンサガーは2次元イジング模型の熱力学的量を数学的に厳密に導出しました。これは統計力学の歴史における金字塔です。

### 1.1 転移温度 $T_c$
無限に広い2次元正方格子において、磁石（強磁性）からバラバラ（常磁性）に変わる温度の境界は以下の式で与えられます：
$$ T_c = \frac{2}{\ln(1+\sqrt{2})} \approx 2.269185 $$
※ ここでは相互作用定数 $J=1$, ボルツマン定数 $k_B=1$ としています。

### 1.2 自発磁化 $M(T)$
温度 $T$ において、外部磁場がない状態で系が持つ磁化（自発磁化）は以下の通りです：
$$ M(T) = \begin{cases} \left[ 1 - \left( \sinh(2/T) \right)^{-4} \right]^{1/8} & (T < T_c) \\ 0 & (T \ge T_c) \end{cases} $$
この式の $1/8$ という指数（臨界指数 $\beta=1/8$）は、転移点付近での磁化の「消え方」の鋭さを表しています。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def get_exact_values():
    Tc = 2.0 / np.log(1.0 + np.sqrt(2.0))
    T_fine = np.linspace(1.5, 3.0, 200)
    M_exact = []
    for T in T_fine:
        if T < Tc:
            val = (1.0 - (np.sinh(2.0/T))**(-4))**(1/8)
        else:
            val = 0.0
        M_exact.append(val)
    return T_fine, M_exact, Tc

T_fine, M_exact, Tc_exact = get_exact_values()
print(f"理論上の転移温度 Tc: {Tc_exact:.6f}")

## 2. シミュレーションの実装（高速版）

比較のために、簡単なメトロポリス法のコードを実行します。

In [ ]:
class IsingSimulation:
    def __init__(self, L, T):
        self.L = L
        self.T = T
        self.spins = np.random.choice([1, -1], size=(L, L))
    
    def step(self):
        L = self.L
        for _ in range(L*L):
            i, j = np.random.randint(0, L, 2)
            dE = 2 * self.spins[i, j] * (
                self.spins[(i+1)%L, j] + self.spins[(i-1)%L, j] +
                self.spins[i, (j+1)%L] + self.spins[i, (j-1)%L]
            )
            if dE <= 0 or np.random.rand() < np.exp(-dE / self.T):
                self.spins[i, j] *= -1

def run_study(Ls, temps, n_steps=1000, n_burnin=300):
    results = {L: {'m': [], 'u': []} for L in Ls}
    for L in Ls:
        for T in temps:
            sim = IsingSimulation(L, T)
            for _ in range(n_burnin): sim.step()
            ms = []
            for _ in range(n_steps):
                sim.step()
                ms.append(np.mean(sim.spins))
            ms = np.array(ms)
            m2 = np.mean(ms**2)
            m4 = np.mean(ms**4)
            results[L]['m'].append(np.mean(np.abs(ms)))
            results[L]['u'].append(1 - m4 / (3 * m2**2))
    return results

Ls = [8, 16]
temps = np.linspace(1.8, 2.8, 15)
data = run_study(Ls, temps)

## 3. データの比較と検証

### 3.1 磁化：有限サイズ効果の可視化
理論上の赤い線（$L=\infty$）に対して、シミュレーション結果（点）を重ねます。

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(T_fine, M_exact, 'r-', label='Exact (L=inf)', alpha=0.7)
for L in Ls:
    plt.plot(temps, data[L]['m'], 'o--', label=f'Sim L={L}')

plt.axvline(Tc_exact, color='black', linestyle=':', label='Exact Tc')
plt.xlabel('Temperature T')
plt.ylabel('Magnetization |M|')
plt.title('Magnetization: Simulation vs Exact Solution')
plt.legend()
plt.grid(True)
plt.show()

#### 考察：なぜズレるのか？
1. **転移が「なだらか」**: 厳密解は $T_c$ で垂直に落ちますが、シミュレーション（有限系）ではサイズ $L$ が小さいほどカーブが緩やかになります。
2. **尾を引く現象 (Tail)**: $T_c$ 以上のバラバラなはずの温度でも、小さな格子では「たまたまスピンが揃う」確率が無視できないため、磁化が完全には 0 になりません。

---

### 3.2 ビンダー累積量：真の $T_c$ を見極める
磁化のズレを解消し、無限系の $T_c$ を特定するためにビンダー累積量プロットを確認します。

In [ ]:
plt.figure(figsize=(10, 6))
for L in Ls:
    plt.plot(temps, data[L]['u'], 'o-', label=f'L={L}')

plt.axvline(Tc_exact, color='red', linestyle='--', label=f'Exact Tc ({Tc_exact:.3f})')
plt.xlabel('Temperature T')
plt.ylabel('Binder Cumulant U_L')
plt.title('Binder Cumulant: Identifying Tc')
plt.legend()
plt.grid(True)
plt.show()

#### 解説：このグラフのすごいところ
- 磁化のグラフでは $L$ ごとにバラバラだった点が、**ビンダー累積量のグラフでは $T_c$ の一点だけで重なります。**
- これにより、たとえ $L=8$ や $16$ といった小さな世界のデータしかなくても、**「もし無限に大きな世界だったら、ここで相転移が起きるはずだ」**という結論（厳密解と同じ値）を導き出すことができます。

## 4. まとめ
1. **厳密解**は、理想的な無限に大きなシステムでの「正解」を教えてくれる。
2. **シミュレーション**は、現実的な有限のシステムでの振る舞いを教えてくれる。
3. **ビンダー累積量**は、その「有限」と「無限」の橋渡しをして、正確な転移点を教えてくれる。

この比較を行うことで、あなたの書いたプログラムが物理的に正しいことを証明できるのです。